In [0]:
%pip install -q scipy

In [0]:
%run ../utils/utils

In [0]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from scipy import stats
 
CAMINHO_METADADOS = "IA/encouders/metadados.json"

##  Ler tudo que o treino já salvou

In [0]:
file_client = container_squad1.get_file_client(CAMINHO_METADADOS)
metadados = json.loads(file_client.download_file().readall().decode("utf-8"))
 
pdf_teste = ler_delta("IA/encouders", "stg_teste_predicoes", STORAGE_OPTIONS).toPandas()
pdf_historico_loss = ler_delta("IA/encouders", "stg_treino_historico_loss", STORAGE_OPTIONS).toPandas().sort_values("epoca")
 
limiar_anomalia = metadados["limiar_anomalia"]
PERCENTIL_LIMIAR = metadados["percentil_limiar"]
 
print("Metadados do treino mais recente:")
print(f"  Limiar de anomalia: {limiar_anomalia:.4f} (percentil {PERCENTIL_LIMIAR})")
print(f"  Melhor val_loss: {metadados['melhor_val_loss']:.4f}")
print(f"  Épocas treinadas: {metadados['qtd_epocas_treinadas']}")
print(f"  Fontes de dados: {metadados['fontes_dados']}")
print(f"\nPedidos no conjunto de teste: {len(pdf_teste)}")

## Visualização dos resultados

In [0]:
fig, eixos = plt.subplots(1, 3, figsize=(18, 5))
 
eixos[0].plot(pdf_historico_loss["epoca"], pdf_historico_loss["loss_treino"], label="loss (treino)")
eixos[0].plot(pdf_historico_loss["epoca"], pdf_historico_loss["val_loss"], label="val_loss (validação)")
eixos[0].set_title("Curva de aprendizado")
eixos[0].set_xlabel("Época")
eixos[0].set_ylabel("MSE")
eixos[0].legend()
 
eixos[1].hist(pdf_teste["erro_reconstrucao"], bins=40, color="#4C72B0", alpha=0.8)
eixos[1].axvline(limiar_anomalia, color="red", linestyle="--", label=f"Limiar (p{PERCENTIL_LIMIAR})")
eixos[1].set_title("Distribuição do erro de reconstrução (teste)")
eixos[1].set_xlabel("Erro de reconstrução")
eixos[1].set_ylabel("Qtd. de pedidos")
eixos[1].legend()
 
contagem = pdf_teste["is_anomaly"].value_counts().reindex([False, True], fill_value=0)
eixos[2].bar(["Normal", "Anomalia"], contagem.values, color=["#4C72B0", "#C44E52"])
eixos[2].set_title(f"Pedidos classificados no teste (n={len(pdf_teste)})")
eixos[2].set_ylabel("Qtd. de pedidos")
for i, v in enumerate(contagem.values):
    eixos[2].text(i, v, str(v), ha="center", va="bottom", fontweight="bold")
 
plt.tight_layout()
plt.show()
 
print(f"\n{contagem[True]} anomalias de {len(pdf_teste)} pedidos no teste "
      f"({100 * contagem[True] / len(pdf_teste):.2f}%).")

##  Análise estatística: Normal vs. Anomalia (média de grupos + teste de hipótese)

In [0]:
grupo_normal = pdf_teste[pdf_teste["is_anomaly"] == False]
grupo_anomalia = pdf_teste[pdf_teste["is_anomaly"] == True]
 
print(f"Grupo Normal: {len(grupo_normal)} | Grupo Anomalia: {len(grupo_anomalia)}")
 
resultados = []
for feat in COLUNAS_NUMERICAS_ANOMALIA:
    valores_normal = grupo_normal[feat].dropna()
    valores_anomalia = grupo_anomalia[feat].dropna()
 
    media_normal = valores_normal.mean()
    media_anomalia = valores_anomalia.mean()
 
    t_stat, p_valor_ttest = stats.ttest_ind(valores_anomalia, valores_normal, equal_var=False, nan_policy="omit")
    u_stat, p_valor_mw = stats.mannwhitneyu(valores_anomalia, valores_normal, alternative="two-sided")
 
    resultados.append({
        "feature": feat,
        "media_normal": round(media_normal, 3),
        "media_anomalia": round(media_anomalia, 3),
        "diferenca_percentual": round(100 * (media_anomalia - media_normal) / media_normal, 1) if media_normal != 0 else None,
        "p_valor_ttest": p_valor_ttest,
        "p_valor_mannwhitney": p_valor_mw,
        "significativo_5pct": (p_valor_ttest < 0.05) and (p_valor_mw < 0.05),
    })
 
df_estatistica = pd.DataFrame(resultados).sort_values("p_valor_ttest")
display(df_estatistica)
 
qtd_sig = df_estatistica["significativo_5pct"].sum()
print(f"\n{qtd_sig} de {len(df_estatistica)} features com diferença estatisticamente significativa (p < 0.05).")
 

##  Desbalanceamento entre squads (diagnóstico)

In [0]:
print("Distribuição por squad no teste:")
print(pdf_teste["origem_squad"].value_counts())
print(pdf_teste["origem_squad"].value_counts(normalize=True).round(4) * 100)
 
print("\nTaxa de anomalia por squad:")
print(
    pdf_teste.groupby("origem_squad")["is_anomaly"]
    .agg(qtd_pedidos="count", qtd_anomalias="sum")
    .assign(taxa_anomalia_pct=lambda df: round(100 * df["qtd_anomalias"] / df["qtd_pedidos"], 2))
)